# HW10-11: Компьютерное зрение в PyTorch
## Часть A: Классификация (CIFAR100)
## Часть B: Детекция (Pascal VOC)

В этом ноутбуке реализованы эксперименты по классификации изображений (CNN, Transfer Learning) и базовая валидация модели детекции объектов.

In [1]:
# Импорт необходимых библиотек
import os
import json
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.utils import draw_bounding_boxes
from PIL import Image

# Настройка воспроизводимости
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Определение устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

# Создание папок для артефактов
os.makedirs("./artifacts/figures", exist_ok=True)

Используемое устройство: cuda


## Часть A: Данные и Трансформы (CIFAR100)

Выбран датасет **CIFAR100**. 
Официального validation split нет, поэтому выделим его из train (80/20).

In [2]:
# Блок №4 - Данные и DataLoader для Части A (CIFAR100)
# С явной фиксацией seed для воспроизводимости split

# Параметры для части A
NUM_CLASSES_A = 100
BATCH_SIZE = 64
NUM_WORKERS = 0  # Для стабильности на Windows в ноутбуке

# Трансформы для простой CNN (C1, C2)
# Базовый трансформ (без аугментаций)
transform_base = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Трансформ с аугментациями (для C2)
transform_aug = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Трансформ для ResNet (требуется нормализация ImageNet)
mean_resnet = [0.485, 0.456, 0.406]
std_resnet = [0.229, 0.224, 0.225]

transform_resnet_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean_resnet, std=std_resnet)
])

transform_resnet_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean_resnet, std=std_resnet)
])

# Загрузка датасета CIFAR100
# Download=True загружает в ~/.torch, что допустимо
train_dataset_full = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_base)
test_dataset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_base)

# Разделение train/val (80/20)
total_size = len(train_dataset_full)
val_size = int(0.2 * total_size)
train_size = total_size - val_size

# фиксация SEED перед разделением
torch.manual_seed(SEED)
np.random.seed(SEED)

train_subset, val_subset = torch.utils.data.random_split(
    train_dataset_full, [train_size, val_size]
)

# Создаём индексы для последовательного использования с разными трансформами
# повторная фиксация SEED для гарантии одинакового split
torch.manual_seed(SEED)
np.random.seed(SEED)

indices = list(range(total_size))
# Перемешиваем индексы с фиксированным seed
torch.manual_seed(SEED)
np.random.seed(SEED)

# Для простоты воспроизведения split используем те же размеры
train_indices = indices[:train_size]
val_indices = indices[train_size:]

# Создание датасетов с разными трансформами но одинаковым split по индексам
train_dataset_cnn = Subset(datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_base), train_indices)
val_dataset_cnn = Subset(datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_base), val_indices)

train_dataset_aug = Subset(datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_aug), train_indices)
# Val для aug используем базовый трансформ (без аугментаций на валидации)
val_dataset_aug = Subset(datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_base), val_indices)

train_dataset_resnet = Subset(datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_resnet_train), train_indices)
val_dataset_resnet = Subset(datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_resnet_val), val_indices)

# DataLoader для CNN (C1, C2)
loader_cnn_base = {
    'train': DataLoader(train_dataset_cnn, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
    'val': DataLoader(val_dataset_cnn, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
    'test': DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
}

loader_cnn_aug = {
    'train': DataLoader(train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
    'val': DataLoader(val_dataset_aug, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
    'test': DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
}

# DataLoader для ResNet (C3, C4)
test_dataset_resnet = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_resnet_val)
loader_resnet = {
    'train': DataLoader(train_dataset_resnet, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
    'val': DataLoader(val_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
    'test': DataLoader(test_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
}

# Sanity check
images, labels = next(iter(loader_cnn_base['train']))
print(f"Shape batch (CNN): {images.shape}, Labels: {labels.shape}")
images_r, labels_r = next(iter(loader_resnet['train']))
print(f"Shape batch (ResNet): {images_r.shape}, Labels: {labels_r.shape}")

i:\Projects\Artificial_Intelligence_Engineer\homeworks\HW08-09\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Shape batch (CNN): torch.Size([64, 3, 32, 32]), Labels: torch.Size([64])
Shape batch (ResNet): torch.Size([64, 3, 224, 224]), Labels: torch.Size([64])


## Часть B: Данные (Pascal VOC Detection)

Выбран трек **Detection**. Датасет **Pascal VOC**.
Используем готовую предобученную модель для инференса.

In [3]:
# Параметры для части B
VOC_YEAR = "2007" # Или 2012, 2007 чаще доступен по умолчанию в torchvision
BATCH_SIZE_B = 1

# Трансформ для VOC (только ToTensor для инференса)
transform_voc = transforms.Compose([
    transforms.ToTensor()
])

# Загрузка датасета
try:
    voc_dataset = datasets.VOCDetection(root="./data_voc", year=VOC_YEAR, image_set="test", download=True, transform=transform_voc)
except Exception as e:
    print(f"Ошибка загрузки VOC: {e}")
    print("Убедитесь, что данные VOC доступны или используйте mock-данные для демонстрации кода.")
    # Для целей выполнения кода без реального скачивания больших данных, 
    # создадим пустой список, если ошибка, но в реальном ДЗ нужно скачать.
    voc_dataset = []

# Если датасет пуст (ошибка загрузки), создадим фейковые данные для прохождения кода
if len(voc_dataset) == 0:
    print("Внимание: Используем синтетические данные для демонстрации логики Part B.")
    # Создадим класс-заглушку
    class MockVOC:
        def __len__(self): return 5
        def __getitem__(self, idx): 
            img = torch.rand(3, 500, 500)
            target = {'boxes': torch.tensor([[10, 10, 100, 100], [200, 200, 300, 300]], dtype=torch.float), 
                      'labels': torch.tensor([1, 2])}
            return img, target
    voc_dataset = MockVOC()

loader_voc = DataLoader(voc_dataset, batch_size=BATCH_SIZE_B, shuffle=False, num_workers=NUM_WORKERS)

## Модели и Функции Обучения

Реализуем простую CNN для экспериментов C1-C2 и функции тренировочного цикла.

In [4]:
# Простая CNN архитектура
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Linear(128 * 4 * 4, num_classes) # 32x32 -> 4x4 после 3 пулингов

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Функции обучения и оценки
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# Функция для сохранения истории
def run_experiment(model, train_loader, val_loader, test_loader, epochs, lr, exp_name, device):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss, v_acc = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            
    # Загрузка лучших весов
    model.load_state_dict(best_model_wts)
    
    # Оценка на тесте (только для лучшей модели в конце)
    _, test_acc = evaluate(model, test_loader, criterion, device)
    
    return model, history, best_val_acc, test_acc

## Эксперименты Части A (C1-C4)

Запустим 4 конфигурации:
- C1: Simple CNN (base)
- C2: Simple CNN (aug)
- C3: ResNet18 (head-only)
- C4: ResNet18 (fine-tune)

In [5]:
# Конфигурация обучения
EPOCHS = 10
LR = 1e-3

results = []

# C1: Simple CNN Base
print("Запуск C1: Simple CNN Base")
model_c1 = SimpleCNN(num_classes=NUM_CLASSES_A)
model_c1, hist_c1, val_acc_c1, _ = run_experiment(  # test_acc не сохраняем
    model_c1, loader_cnn_base['train'], loader_cnn_base['val'], loader_cnn_base['test'], 
    EPOCHS, LR, "C1", device
)
results.append({
    'experiment_id': 'C1', 'task': 'classification', 'dataset': 'CIFAR100', 'seed': SEED,
    'model_summary': 'SimpleCNN', 'optimizer': 'Adam', 'lr': LR, 'epochs_trained': EPOCHS,
    'best_val_accuracy': val_acc_c1, 'test_accuracy': '',  # ПУСТО!
    'precision': '', 'recall': '', 'mean_iou': '', 'notes': 'No augmentations'
})
print(f"C1 Val Acc: {val_acc_c1:.4f}")

Запуск C1: Simple CNN Base
C1 Val Acc: 0.3947


In [6]:
# C2: Simple CNN Aug
print("Запуск C2: Simple CNN Aug")
model_c2 = SimpleCNN(num_classes=NUM_CLASSES_A)
model_c2, hist_c2, val_acc_c2, _ = run_experiment(  # test_acc не сохраняем
    model_c2, loader_cnn_aug['train'], loader_cnn_aug['val'], loader_cnn_base['test'], 
    EPOCHS, LR, "C2", device
)
results.append({
    'experiment_id': 'C2', 'task': 'classification', 'dataset': 'CIFAR100', 'seed': SEED,
    'model_summary': 'SimpleCNN', 'optimizer': 'Adam', 'lr': LR, 'epochs_trained': EPOCHS,
    'best_val_accuracy': val_acc_c2, 'test_accuracy': '',  # ПУСТО!
    'precision': '', 'recall': '', 'mean_iou': '', 'notes': 'With augmentations'
})
print(f"C2 Val Acc: {val_acc_c2:.4f}")

Запуск C2: Simple CNN Aug
C2 Val Acc: 0.4227


In [7]:
# C3: ResNet18 Head Only
print("Запуск C3: ResNet18 Head Only")
model_c3 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model_c3.fc.in_features
model_c3.fc = nn.Linear(num_ftrs, NUM_CLASSES_A)

for param in model_c3.parameters():
    param.requires_grad = False
for param in model_c3.fc.parameters():
    param.requires_grad = True

model_c3, hist_c3, val_acc_c3, _ = run_experiment(  # test_acc не сохраняем
    model_c3, loader_resnet['train'], loader_resnet['val'], loader_resnet['test'], 
    EPOCHS, LR, "C3", device
)
results.append({
    'experiment_id': 'C3', 'task': 'classification', 'dataset': 'CIFAR100', 'seed': SEED,
    'model_summary': 'ResNet18', 'optimizer': 'Adam', 'lr': LR, 'epochs_trained': EPOCHS,
    'best_val_accuracy': val_acc_c3, 'test_accuracy': '',  # ПУСТО!
    'precision': '', 'recall': '', 'mean_iou': '', 'notes': 'Frozen backbone'
})
print(f"C3 Val Acc: {val_acc_c3:.4f}")

Запуск C3: ResNet18 Head Only
C3 Val Acc: 0.5852


In [8]:
# C4: ResNet18 Fine-tune
print("Запуск C4: ResNet18 Fine-tune")
model_c4 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model_c4.fc.in_features
model_c4.fc = nn.Linear(num_ftrs, NUM_CLASSES_A)

for param in model_c4.parameters():
    param.requires_grad = False
for param in model_c4.layer4.parameters():
    param.requires_grad = True
for param in model_c4.fc.parameters():
    param.requires_grad = True

model_c4, hist_c4, val_acc_c4, _ = run_experiment(  # test_acc не сохраняем
    model_c4, loader_resnet['train'], loader_resnet['val'], loader_resnet['test'], 
    EPOCHS, LR, "C4", device
)
results.append({
    'experiment_id': 'C4', 'task': 'classification', 'dataset': 'CIFAR100', 'seed': SEED,
    'model_summary': 'ResNet18', 'optimizer': 'Adam', 'lr': LR, 'epochs_trained': EPOCHS,
    'best_val_accuracy': val_acc_c4, 'test_accuracy': '',  # ПУСТО! (заполним позже для лучшего)
    'precision': '', 'recall': '', 'mean_iou': '', 'notes': 'Fine-tune layer4+fc'
})
print(f"C4 Val Acc: {val_acc_c4:.4f}")

Запуск C4: ResNet18 Fine-tune
C4 Val Acc: 0.6798


In [9]:
# Блок №14 [py] - Выбор лучшей модели и ОДНА оценка на test

# Выбор лучшей модели по val_accuracy
df_results = pd.DataFrame(results)
best_row = df_results.loc[df_results['best_val_accuracy'].idxmax()]
best_exp_id = best_row['experiment_id']

print(f"\nЛучший эксперимент части A: {best_exp_id} с Val Acc: {best_row['best_val_accuracy']:.4f}")

# Оцениваем НА TEST ТОЛЬКО ЛУЧШУЮ МОДЕЛЬ (один раз!)
if best_exp_id == 'C1':
    best_model = model_c1
    _, test_acc_best = evaluate(best_model, loader_cnn_base['test'], nn.CrossEntropyLoss(), device)
    config_note = "SimpleCNN, base transforms"
elif best_exp_id == 'C2':
    best_model = model_c2
    _, test_acc_best = evaluate(best_model, loader_cnn_base['test'], nn.CrossEntropyLoss(), device)
    config_note = "SimpleCNN, aug transforms"
elif best_exp_id == 'C3':
    best_model = model_c3
    _, test_acc_best = evaluate(best_model, loader_resnet['test'], nn.CrossEntropyLoss(), device)
    config_note = "ResNet18, head-only"
else:  # C4
    best_model = model_c4
    _, test_acc_best = evaluate(best_model, loader_resnet['test'], nn.CrossEntropyLoss(), device)
    config_note = "ResNet18, fine-tune"

print(f"Test Accuracy лучшей модели ({best_exp_id}): {test_acc_best:.4f}")

# Обновляем test_accuracy ТОЛЬКО для лучшей модели в results
for i, exp in enumerate(results):
    if exp['experiment_id'] == best_exp_id:
        results[i]['test_accuracy'] = float(test_acc_best)

# Сохранение лучшей модели
torch.save(best_model.state_dict(), "./artifacts/best_classifier.pt")

# Сохранение конфига
config_dict = {
    "dataset": "CIFAR100",
    "model": best_row['model_summary'],
    "transforms": config_note,
    "optimizer": best_row['optimizer'],
    "lr": float(best_row['lr']),
    "epochs": int(best_row['epochs_trained']),
    "seed": SEED,
    "best_val_accuracy": float(best_row['best_val_accuracy']),
    "test_accuracy": float(test_acc_best)
}
with open("./artifacts/best_classifier_config.json", "w") as f:
    json.dump(config_dict, f, indent=4)


Лучший эксперимент части A: C4 с Val Acc: 0.6798
Test Accuracy лучшей модели (C4): 0.6809


## Часть B: Детекция (V1-V2)

Используем pretrained FasterRCNN. 
Режим V1: threshold=0.3
Режим V2: threshold=0.7

In [10]:
import xml.etree.ElementTree as ET

# Параметры для части B
BATCH_SIZE_B = 1
NUM_WORKERS = 0

# Трансформ для VOC
transform_voc = transforms.Compose([
    transforms.ToTensor()
])

# VOC классы (для маппинга названий в номера)
VOC_CLASSES = [
    '__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 
    'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 
    'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

def parse_voc_annotation(target):
    """
    Парсит XML аннотацию VOC и возвращает словарь с boxes и labels.
    """
    annotation = target['annotation']
    objects = annotation['object']
    
    if not isinstance(objects, list):
        objects = [objects]
    
    boxes = []
    labels = []
    
    for obj in objects:
        # Bounding box
        bbox = obj['bndbox']
        xmin = float(bbox['xmin'])
        ymin = float(bbox['ymin'])
        xmax = float(bbox['xmax'])
        ymax = float(bbox['ymax'])
        boxes.append([xmin, ymin, xmax, ymax])
        
        # Label (название класса -> номер)
        class_name = obj['name']
        if class_name in VOC_CLASSES:
            label = VOC_CLASSES.index(class_name)
        else:
            label = 0  # background
        labels.append(label)
    
    return {
        'boxes': torch.as_tensor(boxes, dtype=torch.float32),
        'labels': torch.as_tensor(labels, dtype=torch.int64)
    }


# Загрузка датасета
try:
    voc_dataset_raw = datasets.VOCDetection(
        root="./data_voc", 
        year="2007", 
        image_set="test", 
        download=True, 
        transform=transform_voc
    )
    print(f"VOC датасет загружен: {len(voc_dataset_raw)} изображений")
    
    # Оборачиваем для правильного формата target
    class VOCDetectionWrapper:
        def __init__(self, raw_dataset):
            self.raw_dataset = raw_dataset
        
        def __len__(self):
            return len(self.raw_dataset)
        
        def __getitem__(self, idx):
            img, target = self.raw_dataset[idx]
            target_parsed = parse_voc_annotation(target)
            return img, target_parsed
    
    voc_dataset = VOCDetectionWrapper(voc_dataset_raw)
    
except Exception as e:
    print(f"Ошибка загрузки VOC: {e}")
    print("Используем синтетические данные для демонстрации логики Part B.")
    
    class MockVOC:
        def __len__(self): 
            return 5
        def __getitem__(self, idx): 
            img = torch.rand(3, 500, 500)
            target = {
                'boxes': torch.tensor([[10, 10, 100, 100], [200, 200, 300, 300]], dtype=torch.float), 
                'labels': torch.tensor([1, 2])
            }
            return img, target
    
    voc_dataset = MockVOC()


# Custom collate_fn для detection
def collate_fn(batch):
    images = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    return images, targets

loader_voc = DataLoader(
    voc_dataset, 
    batch_size=BATCH_SIZE_B, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

print(f"DataLoader VOC создан: {len(loader_voc)} батчей")

VOC датасет загружен: 4952 изображений
DataLoader VOC создан: 4952 батчей


In [11]:
def compute_iou(box1, box2):
    """
    Вычисляет IoU для двух bounding boxes.
    Формат: [x1, y1, x2, y2]
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - inter_area
    if union_area == 0:
        return 0.0
    return inter_area / union_area


def evaluate_detection(loader, model, score_thresh, device, max_samples=5):
    """
    Оценка модели детекции с расчётом precision, recall, mean IoU.
    """
    precisions = []
    recalls = []
    ious = []
    vis_images = []
    
    count = 0
    for imgs, targets in loader:
        if count >= max_samples: 
            break
        
        img = imgs[0].to(device)
        target = targets[0]
        
        # Проверка наличия ключей
        if 'boxes' not in target or len(target['boxes']) == 0:
            count += 1
            continue
        
        gt_boxes = target['boxes'].to(device)
        
        # Инференс
        with torch.no_grad():
            prediction = model([img])
        
        pred_boxes = prediction[0]['boxes']
        pred_scores = prediction[0]['scores']
        pred_labels = prediction[0]['labels']
        
        # Фильтрация по порогу уверенности
        keep = pred_scores >= score_thresh
        pred_boxes = pred_boxes[keep]
        pred_scores = pred_scores[keep]
        
        # Визуализация
        img_uint8 = (img.cpu() * 255).to(torch.uint8)
        if len(pred_boxes) > 0:
            vis_img = draw_bounding_boxes(img_uint8, pred_boxes.cpu(), colors="yellow", width=3)
        else:
            vis_img = img_uint8
        vis_images.append(vis_img)
        
        # Расчёт метрик
        tp = 0
        fp = 0
        sample_ious = []
        
        for gt_box in gt_boxes:
            best_iou = 0
            for pb in pred_boxes:
                iou = compute_iou(gt_box.cpu().numpy(), pb.cpu().numpy())
                if iou > best_iou:
                    best_iou = iou
            if best_iou >= 0.5:
                tp += 1
                sample_ious.append(best_iou)
            else:
                fp += 1
        
        if len(gt_boxes) > 0:
            recalls.append(tp / len(gt_boxes))
        else:
            recalls.append(0.0)
            
        if (tp + fp) > 0:
            precisions.append(tp / (tp + fp))
        else:
            precisions.append(0.0)
            
        if len(sample_ious) > 0:
            ious.append(np.mean(sample_ious))
        else:
            ious.append(0.0)
            
        count += 1
        
    # Защита от пустых списков
    if len(precisions) == 0:
        precisions = [0.0]
    if len(recalls) == 0:
        recalls = [0.0]
    if len(ious) == 0:
        ious = [0.0]
        
    return np.mean(precisions), np.mean(recalls), np.mean(ious), vis_images


# Загрузка модели детекции
print("Загрузка FasterRCNN...")
model_det = models.detection.fasterrcnn_resnet50_fpn(
    weights=models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
)
model_det.to(device)
model_det.eval()
print("Модель загружена и переведена в eval режим")


# Режим V1
print("\nЗапуск V1 (threshold=0.3)")
p1, r1, iou1, vis_v1 = evaluate_detection(loader_voc, model_det, 0.3, device)
print(f"V1 - Precision: {p1:.4f}, Recall: {r1:.4f}, Mean IoU: {iou1:.4f}")

# Режим V2
print("\nЗапуск V2 (threshold=0.7)")
p2, r2, iou2, vis_v2 = evaluate_detection(loader_voc, model_det, 0.7, device)
print(f"V2 - Precision: {p2:.4f}, Recall: {r2:.4f}, Mean IoU: {iou2:.4f}")

# Добавление в results
results.append({
    'experiment_id': 'V1', 'task': 'detection', 'dataset': 'PascalVOC', 'seed': SEED,
    'model_summary': 'FasterRCNN', 'optimizer': '', 'lr': '', 'epochs_trained': 0,
    'best_val_accuracy': '', 'test_accuracy': '',
    'precision': float(p1), 'recall': float(r1), 'mean_iou': float(iou1), 
    'notes': 'threshold=0.3'
})
results.append({
    'experiment_id': 'V2', 'task': 'detection', 'dataset': 'PascalVOC', 'seed': SEED,
    'model_summary': 'FasterRCNN', 'optimizer': '', 'lr': '', 'epochs_trained': 0,
    'best_val_accuracy': '', 'test_accuracy': '',
    'precision': float(p2), 'recall': float(r2), 'mean_iou': float(iou2), 
    'notes': 'threshold=0.7'
})

# Визуализация детекции
if len(vis_v1) > 0:
    fig, axes = plt.subplots(1, len(vis_v1), figsize=(15, 5))
    if len(vis_v1) == 1: 
        axes = [axes]
    for ax, img in zip(axes, vis_v1):
        ax.imshow(img.permute(1, 2, 0))
        ax.axis('off')
    plt.suptitle("Detection Examples V1 (thresh=0.3)")
    plt.savefig("./artifacts/figures/detection_examples.png")
    plt.close()
    print("Визуализация детекции сохранена: ./artifacts/figures/detection_examples.png")

# График метрик детекции
plt.figure(figsize=(8, 5))
x = ['V1 (0.3)', 'V2 (0.7)']
prec = [p1, p2]
rec = [r1, r2]
x_pos = np.arange(len(x))
plt.bar(x_pos - 0.2, prec, 0.4, label='Precision')
plt.bar(x_pos + 0.2, rec, 0.4, label='Recall')
plt.xticks(x_pos, x)
plt.legend()
plt.title("Detection Metrics Comparison")
plt.savefig("./artifacts/figures/detection_metrics.png")
plt.close()
print("График метрик сохранён: ./artifacts/figures/detection_metrics.png")

Загрузка FasterRCNN...
Модель загружена и переведена в eval режим

Запуск V1 (threshold=0.3)
V1 - Precision: 0.8500, Recall: 0.8500, Mean IoU: 0.8873

Запуск V2 (threshold=0.7)
V2 - Precision: 0.7500, Recall: 0.7500, Mean IoU: 0.8717
Визуализация детекции сохранена: ./artifacts/figures/detection_examples.png
График метрик сохранён: ./artifacts/figures/detection_metrics.png


In [12]:
# Блок №17 [py] - Сохранение runs.csv и итоговых артефактов

import pandas as pd
import json

# Сохранение итоговой таблицы runs.csv
df_final = pd.DataFrame(results)
df_final.to_csv("./artifacts/runs.csv", index=False)
print("runs.csv сохранён: ./artifacts/runs.csv")

# Проверка наличия всех артефактов
import os

required_files = [
    "./artifacts/runs.csv",
    "./artifacts/best_classifier.pt",
    "./artifacts/best_classifier_config.json",
    "./artifacts/figures/classification_curves_best.png",
    "./artifacts/figures/classification_compare.png",
    "./artifacts/figures/augmentations_preview.png",
    "./artifacts/figures/detection_examples.png",
    "./artifacts/figures/detection_metrics.png"
]

print("\n Проверка артефактов:")
for file in required_files:
    exists = os.path.exists(file)
    status = "OK" if exists else "NOT OK"
    print(f"{status} {file}")

# Вывод таблицы результатов
print("\n Итоговая таблица результатов:")
print(df_final[['experiment_id', 'task', 'best_val_accuracy', 'test_accuracy', 'precision', 'recall', 'mean_iou']].to_string(index=False))

runs.csv сохранён: ./artifacts/runs.csv

 Проверка артефактов:
OK ./artifacts/runs.csv
OK ./artifacts/best_classifier.pt
OK ./artifacts/best_classifier_config.json
OK ./artifacts/figures/classification_curves_best.png
OK ./artifacts/figures/classification_compare.png
OK ./artifacts/figures/augmentations_preview.png
OK ./artifacts/figures/detection_examples.png
OK ./artifacts/figures/detection_metrics.png

 Итоговая таблица результатов:
experiment_id           task best_val_accuracy test_accuracy precision recall  mean_iou
           C1 classification            0.3947                                         
           C2 classification            0.4227                                         
           C3 classification            0.5852                                         
           C4 classification            0.6798        0.6809                           
           V1      detection                                      0.85   0.85  0.887336
           V2      detection    

## Заключение

Все эксперименты выполнены. Артефакты сохранены в папку `artifacts`. 
Отчёт заполняется в файле `report.md` на основе полученных метрик.